In [1]:
import pandas as pd
import os

# ==========================================
# 1. ファイル名の定義 (アップロードされたファイル名)
# ==========================================
# 統合データ (MDS座標 + PCEなど)
FILE_THESIS_A = 'ThesisData_A_OH_plus_others_MDS_Targets (1).csv'
FILE_THESIS_B = 'ThesisData_B_FP_plus_others_MDS_Targets.csv'

# クラスタリング結果一覧 (ラベル)
FILE_LABELS_A = 'cluster_labels_matrix_samples_A_OH_plus_others_20251130_153500.csv'
FILE_LABELS_B = 'cluster_labels_matrix_samples_B_FP_plus_others_20251130_153500.csv'

# 詳細解析結果 (中身の材料名など)
FILE_DETAIL_OH_K50 = 'Detail_OH_Clusters_linear_top3_k50.csv'
FILE_DETAIL_OH_K10 = 'Detail_OH_Clusters_linear_top3_k10.csv'
FILE_DETAIL_FP_K50 = 'Detail_FP_Clusters_linear_top3_k50.csv'
FILE_DETAIL_FP_K10 = 'Detail_FP_Clusters_linear_top3_k10.csv'

# 出力ファイル名
OUT_TABLE_MAIN  = 'Table_4_x_Sweet_Spot_Summary.csv'
OUT_TABLE_OH_ALL = 'Table_Appendix_OH_All_Clusters_k50.csv'
OUT_TABLE_FP_ALL = 'Table_Appendix_FP_All_Clusters_k50.csv'

# ==========================================
# 2. データ読み込み & 結合
# ==========================================
def load_and_merge(f_thesis, f_labels):
    """MDSデータとクラスタラベルをIDで結合する"""
    if not os.path.exists(f_thesis) or not os.path.exists(f_labels):
        print(f"[WARN] Files not found: {f_thesis} or {f_labels}")
        return None
    
    try:
        df_t = pd.read_csv(f_thesis)
        # R由来の Unnamed: 0 を ID にリネーム
        if 'Unnamed: 0' in df_t.columns:
            df_t.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
        
        df_l = pd.read_csv(f_labels)
        # IDで結合
        df = pd.merge(df_t, df_l, on='ID', how='inner')
        return df
    except Exception as e:
        print(f"[ERROR] Merge failed: {e}")
        return None

def load_detail(path):
    """詳細解析ファイルを読み込む"""
    if not os.path.exists(path): return None
    return pd.read_csv(path)

# ==========================================
# 3. メイン処理
# ==========================================

print("Loading Data...")
df_A = load_and_merge(FILE_THESIS_A, FILE_LABELS_A) # OH
df_B = load_and_merge(FILE_THESIS_B, FILE_LABELS_B) # FP

det_OH_k50 = load_detail(FILE_DETAIL_OH_K50)
det_OH_k10 = load_detail(FILE_DETAIL_OH_K10)
det_FP_k50 = load_detail(FILE_DETAIL_FP_K50)
det_FP_k10 = load_detail(FILE_DETAIL_FP_K10)

# --- Helper: クラスターごとの統計量を計算 ---
def get_stats(df, col_k, cluster_id, detail_df, type_name):
    # 該当クラスターのデータを抽出
    subset = df[df[col_k] == cluster_id]
    pce_data = subset['PCEmax']
    
    # 主要成分の取得
    desc = "N/A"
    if detail_df is not None:
        row = detail_df[detail_df['Cluster_ID'] == cluster_id]
        if not row.empty:
            col_comp = 'Major_Material' if type_name == 'OH' else 'Major_Fragment'
            raw_name = str(row[col_comp].values[0])
            # 表を見やすくするための簡易置換
            desc = raw_name.replace("SMILESsname", "").replace("p1M_", "").replace("nM_", "").replace("inM_", "")
    
    return {
        'Count': len(subset),
        'Mean_PCE': pce_data.mean(),
        'Std_Dev': pce_data.std(),
        'Max_PCE': pce_data.max(),
        'Major_Component': desc
    }

# ---------------------------------------------------------
# A. Table 4.x: Sweet Spot Summary (成功領域の要約)
# ---------------------------------------------------------
if df_A is not None and df_B is not None:
    print("\nGenerating Table 4.x (Sweet Spot Summary)...")
    rows = []
    
    # 1. OH (k=10) Best
    col = 'linear_top3_k10'
    best_id = df_A.groupby(col)['PCEmax'].mean().idxmax()
    st = get_stats(df_A, col, best_id, det_OH_k10, 'OH')
    rows.append(['OH (Material)', 10, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 2. OH (k=50) Best
    col = 'linear_top3_k50'
    best_id = df_A.groupby(col)['PCEmax'].mean().idxmax()
    st = get_stats(df_A, col, best_id, det_OH_k50, 'OH')
    rows.append(['OH (Material)', 50, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 3. FP (k=10) Best
    col = 'linear_top3_k10'
    best_id = df_B.groupby(col)['PCEmax'].mean().idxmax()
    st = get_stats(df_B, col, best_id, det_FP_k10, 'FP')
    rows.append(['FP (Structure)', 10, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 4. FP (k=50) Best
    col = 'linear_top3_k50'
    best_id = df_B.groupby(col)['PCEmax'].mean().idxmax()
    st = get_stats(df_B, col, best_id, det_FP_k50, 'FP')
    rows.append(['FP (Structure)', 50, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # DataFrame化 & 保存
    cols = ['Dataset', 'Resolution (k)', 'Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component']
    df_table4x = pd.DataFrame(rows, columns=cols)
    df_table4x.to_csv(OUT_TABLE_MAIN, index=False)
    print(f"  -> Saved: {OUT_TABLE_MAIN}")
    print(df_table4x)


# ---------------------------------------------------------
# B. Table Appendix: 全クラスターランキング (k=50)
# ---------------------------------------------------------
def generate_full_ranking(df, detail_df, type_name, out_path):
    print(f"\nGenerating Full Ranking for {type_name} (k=50)...")
    col_k50 = 'linear_top3_k50'
    
    # 全クラスターIDを取得
    cluster_ids = sorted(df[col_k50].unique())
    
    rank_rows = []
    for cid in cluster_ids:
        st = get_stats(df, col_k50, cid, detail_df, type_name)
        rank_rows.append([cid, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
        
    # DataFrame化
    cols = ['Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component']
    df_rank = pd.DataFrame(rank_rows, columns=cols)
    
    # PCE平均で降順ソート (ランキング)
    df_rank = df_rank.sort_values('Mean PCE (%)', ascending=False)
    
    # 保存
    df_rank.to_csv(out_path, index=False)
    print(f"  -> Saved: {out_path}")
    # 上位5つを表示
    print(df_rank.head(5))

# OHランキング作成
if df_A is not None and det_OH_k50 is not None:
    generate_full_ranking(df_A, det_OH_k50, 'OH', OUT_TABLE_OH_ALL)

# FPランキング作成
if df_B is not None and det_FP_k50 is not None:
    generate_full_ranking(df_B, det_FP_k50, 'FP', OUT_TABLE_FP_ALL)

print("\n✅ All Tables Generated Successfully!")


Loading Data...
[WARN] Files not found: ThesisData_A_OH_plus_others_MDS_Targets (1).csv or cluster_labels_matrix_samples_A_OH_plus_others_20251130_153500.csv
[WARN] Files not found: ThesisData_B_FP_plus_others_MDS_Targets.csv or cluster_labels_matrix_samples_B_FP_plus_others_20251130_153500.csv

✅ All Tables Generated Successfully!
